# ⚖️ Hypothesis Testing & p-values
## *Did the Student Cheat — or Just Get Lucky?*

---

## 🏛️ The Setup

A student takes a **10-question True/False exam** and scores **9 out of 10**.

The teacher is suspicious. The student says:

> *"I didn't cheat! I just guessed every answer randomly!"*

The teacher takes the student to the **principal's office** — which works exactly like a courtroom.

The principal must decide: **Did the student cheat, or did they just get lucky?**

---

### ⚖️ The Courtroom Rules

The principal follows one golden rule:

> **"The student is INNOCENT until proven guilty."**

This means: we start by *assuming* the student was guessing randomly. We only change our mind if the evidence is strong enough.

| Courtroom | Our Problem |
|-----------|-------------|
| Defendant | The student |
| "Innocent" | Student was guessing randomly |
| "Guilty" | Student was cheating |
| Evidence | The score: 9 out of 10 |
| Judge | Statistical test |
| "Beyond reasonable doubt" | Significance level α |

This is **exactly** how hypothesis testing works. Let's go step by step. 🚀

In [ ]:
# Setup — run this first!
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#F5F5F5',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size':  12,
    'axes.titlesize': 13,
})
print("✅ Ready! Let's figure out if the student cheated.")

---

## Step 1 — State the Hypotheses

Before looking at any evidence, the principal must clearly state two opposing positions.

---

### H₀ — The Null Hypothesis: "Student is Innocent"

> **H₀: The student was guessing randomly. Each answer had a 50% chance of being correct.**

This is our **starting assumption**. We give the student the benefit of the doubt.

If the student truly guessed randomly on 10 True/False questions, we'd expect them to get around **5 out of 10** correct on average — just like flipping a coin 10 times.

---

### H₁ — The Alternative Hypothesis: "Student is Guilty"

> **H₁: The student was cheating. They knew the answers and scored higher than random chance would explain.**

This is what the teacher believes. But the teacher must **prove it** — the student doesn't have to prove their innocence.

---

```
H₀ (Innocent): Student guesses randomly → Expected score ≈ 5/10
H₁ (Guilty):   Student was cheating    → Score will be suspiciously high
```

> 💡 We will ASSUME H₀ is true and ask:
> *"If the student truly guessed randomly, how surprising is a score of 9?"*

In [ ]:
# ── Step 1 Visual: Two possible worlds ──────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Two Possible Worlds', fontsize=15, fontweight='bold')

scores = np.arange(0, 11)

# H0 world: guessing (p=0.5)
ax1 = axes[0]
h0_probs = stats.binom.pmf(scores, n=10, p=0.5)
ax1.bar(scores, h0_probs, color='#4A90D9', alpha=0.8, edgecolor='white', linewidth=1.5)
ax1.axvline(x=5, color='navy', linestyle='--', linewidth=2, label='Expected = 5')
ax1.set_title('H₀ World: Student is INNOCENT\n(Guessing randomly, 50/50)', fontweight='bold')
ax1.set_xlabel('Score out of 10')
ax1.set_ylabel('Probability')
ax1.set_xticks(scores)
ax1.legend()
ax1.text(0.5, 0.82, '"I just guessed!"\nExpect scores around 5.',
         transform=ax1.transAxes, ha='center', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='#D6EAF8', alpha=0.9))

# H1 world: cheating (p=0.9)
ax2 = axes[1]
h1_probs = stats.binom.pmf(scores, n=10, p=0.9)
ax2.bar(scores, h1_probs, color='#E74C3C', alpha=0.8, edgecolor='white', linewidth=1.5)
ax2.axvline(x=9, color='darkred', linestyle='--', linewidth=2, label='Expected ≈ 9')
ax2.set_title('H₁ World: Student is GUILTY\n(Cheating — knows the answers)', fontweight='bold')
ax2.set_xlabel('Score out of 10')
ax2.set_ylabel('Probability')
ax2.set_xticks(scores)
ax2.legend()
ax2.text(0.5, 0.82, '"They cheated!"\nExpect scores of 9 or 10.',
         transform=ax2.transAxes, ha='center', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='#FADBD8', alpha=0.9))

plt.tight_layout()
plt.show()

print("Our student scored 9/10.")
print("Which world does that look like it came from?")
print("That's what hypothesis testing figures out!")

---

## Step 2 — Set the Threshold (Significance Level α)

Before looking at evidence, the principal must decide:

> **"How suspicious does the evidence need to be before I say the student cheated?"**

In court this is called **"beyond reasonable doubt"**.
In statistics it is called the **significance level**, written as **α (alpha)**.

The most common choice in science is:

### α = 0.05

> *"If a truly innocent (guessing) student would only produce this score — or higher — less than 5% of the time, then we say it's suspicious enough to reject the innocent explanation."*

**You must set α BEFORE looking at the data** — just like a judge sets the rules before a trial begins.

---

```
α = 0.05  means:  "I need the evidence to be in the top 5% most suspicious
                   before I'll call the student guilty."
```

---

## Step 3 — Build the Null Distribution
### *"What does an innocent student look like?"*

The principal asks a very important question before judging:

> **"If a student genuinely guessed randomly on every question, what scores would we normally see?"**

To answer this, let's imagine **10,000 different students** who all guess randomly — every answer is a 50/50 coin flip. What scores do they get?

This gives us a picture of **innocent behavior**. It's our baseline for comparison.

In [ ]:
# ── Step 3: Simulate 10,000 innocent (guessing) students ────

np.random.seed(42)
n_students  = 10_000
n_questions = 10

# Each student guesses randomly — 50% chance per question
innocent_scores = np.random.binomial(n=n_questions, p=0.5, size=n_students)

print("📊 10,000 Innocent (Randomly Guessing) Students:")
print(f"   Average score : {innocent_scores.mean():.2f}  (expected: 5.0)")
print(f"   Lowest score  : {innocent_scores.min()}")
print(f"   Highest score : {innocent_scores.max()}")
print()
print("   Score breakdown:")
print(f"   {'Score':>7} | {'Count':>7} | {'% of students':>15}")
print("   " + "-" * 35)
for s in range(11):
    c = np.sum(innocent_scores == s)
    print(f"   {s:>7} | {c:>7,} | {c/n_students*100:>13.1f}%")

In [ ]:
# ── Step 3 Visual: The Null Distribution ────────────────────

fig, ax = plt.subplots(figsize=(11, 6))

score_vals = np.arange(0, 11)
counts     = [np.sum(innocent_scores == s) for s in score_vals]

ax.bar(score_vals, counts, color='#4A90D9', alpha=0.85,
       edgecolor='white', linewidth=2, width=0.7)

# Label each bar
for s, c in zip(score_vals, counts):
    ax.text(s, c + 40, f'{c:,}', ha='center', fontsize=8.5,
            fontweight='bold', color='#2C3E50')

ax.axvline(x=5, color='navy', linestyle='--', linewidth=2.5,
           label='Average = 5  (what we expect from guessing)')

ax.set_xlabel('Score on 10-Question Exam', fontsize=12)
ax.set_ylabel('Number of Students (out of 10,000)', fontsize=12)
ax.set_title('The Null Distribution\n'
             'What scores do INNOCENT (randomly guessing) students get?',
             fontsize=13, fontweight='bold')
ax.set_xticks(score_vals)
ax.legend(fontsize=10)

ax.text(0.02, 0.97,
        '⚖️ This is our picture of innocence.\n'
        'Most students score 3 – 7.\n'
        'Scoring 9 or 10 is rare — but\n'
        'does happen by luck sometimes!',
        transform=ax.transAxes, va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='#E8F4FD', alpha=0.95))

plt.tight_layout()
plt.show()

---

## Step 4 — Look at the Observed Evidence

Our student scored **9 out of 10**.

The principal now asks:

> **"Out of all the innocent students we just simulated, how many scored 9 or higher?"**

If very few innocent students score that high, then a score of 9 is **suspicious evidence** — it's the kind of result that cheaters produce, not innocent guessers.

Let's mark this on our chart.

In [ ]:
# ── Step 4 Visual: Mark the observed score ──────────────────

observed_score = 9

fig, ax = plt.subplots(figsize=(11, 6))

for s, c in zip(score_vals, counts):
    color = '#E74C3C' if s >= observed_score else '#4A90D9'
    ax.bar(s, c, color=color, alpha=0.85, edgecolor='white', linewidth=2, width=0.7)
    ax.text(s, c + 40, f'{c:,}', ha='center', fontsize=8.5,
            fontweight='bold', color='#2C3E50')

ax.axvline(x=observed_score - 0.5, color='darkred', linestyle='-',
           linewidth=3, label=f'Our student scored {observed_score}/10', zorder=10)

suspicious_count = np.sum(innocent_scores >= observed_score)
ax.annotate(
    f'🚨 Only {suspicious_count} innocent students\nout of 10,000 scored {observed_score} or higher\n({suspicious_count/100:.1f}% of innocent students)',
    xy=(9.5, counts[9] + 10), xytext=(7.0, 1800),
    fontsize=10, color='#C0392B', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#C0392B', lw=2),
    bbox=dict(boxstyle='round', facecolor='#FADBD8', alpha=0.9)
)

blue_p = mpatches.Patch(color='#4A90D9', alpha=0.85, label='Normal innocent scores')
red_p  = mpatches.Patch(color='#E74C3C', alpha=0.85, label=f'Scores as high as our student (≥ {observed_score})')
ax.legend(handles=[blue_p, red_p], fontsize=10)

ax.set_xlabel('Score on 10-Question Exam', fontsize=12)
ax.set_ylabel('Number of Students (out of 10,000)', fontsize=12)
ax.set_title(f'Our student scored {observed_score}/10 — How suspicious is that?',
             fontsize=13, fontweight='bold')
ax.set_xticks(score_vals)

plt.tight_layout()
plt.show()

print(f"Out of 10,000 innocent (guessing) students:")
print(f"  → {suspicious_count} scored {observed_score} or higher")
print(f"  → That's {suspicious_count/100:.1f}% of innocent students")
print(f"\nThis {suspicious_count/100:.1f}% is our p-value!")

---

## Step 5 — Calculate the p-value

The **p-value** answers one specific question:

> **"If the student were truly innocent (guessing randomly), what is the probability they would score this high — or higher — just by luck?"**

From our simulation:
- 10,000 innocent students were simulated
- We count how many scored **9 or higher**
- p-value = that count ÷ 10,000

---

### ⚖️ Courtroom Translation

> *"If the defendant were truly innocent, how likely is it that they would produce evidence this incriminating — or more — purely by chance?"*

**Small p-value** → Even innocent students rarely score this high → **Suspicious!**  
**Large p-value** → Innocent students score this high all the time → **Not suspicious**

In [ ]:
# ── Step 5: Calculate the p-value ───────────────────────────

# From simulation
p_value_sim   = np.sum(innocent_scores >= observed_score) / n_students

# Exact theoretical answer
p_value_exact = 1 - stats.binom.cdf(observed_score - 1, n=n_questions, p=0.5)

print("=" * 52)
print("   📐 P-VALUE CALCULATION")
print("=" * 52)
print()
print(f"   Our student's score:   {observed_score} / {n_questions}")
print()
print(f"   From simulation:")
print(f"   {np.sum(innocent_scores >= observed_score)} out of 10,000 innocent")
print(f"   students scored {observed_score} or higher.")
print()
print(f"   p-value (simulation) = {p_value_sim:.4f}")
print(f"   p-value (exact)      = {p_value_exact:.4f}")
print()
print("=" * 52)
print()
print(f"   ⚖️  In plain English:")
print(f"   'If this student truly guessed randomly,")
print(f"    there is only a {p_value_exact*100:.1f}% chance they would")
print(f"    score {observed_score}/10 or higher just by luck.'")
print()
print(f"   That is a very small probability.")
print(f"   Next step: is it small ENOUGH to call them guilty?")

In [ ]:
# ── Step 5 Visual: p-value on the distribution ──────────────

fig, ax = plt.subplots(figsize=(11, 6))

for s, c in zip(score_vals, counts):
    color = '#E74C3C' if s >= observed_score else '#4A90D9'
    ax.bar(s, c, color=color, alpha=0.85, edgecolor='white', linewidth=2, width=0.7)

ax.axvline(x=observed_score - 0.5, color='darkred', linestyle='-', linewidth=3)

# p-value label
pval_count = np.sum(innocent_scores >= observed_score)
ax.text(9.5, 650,
        f'p-value\n= {pval_count} / 10,000\n= {p_value_exact:.4f}\n({p_value_exact*100:.1f}%)',
        ha='center', fontsize=11, color='#C0392B', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#FADBD8', alpha=0.95))

# What the p-value means
ax.text(0.02, 0.97,
        '⚖️  The p-value is the RED area.\n\n'
        'It answers: "How often does an\n'
        'innocent student score THIS high?"\n\n'
        f'Answer: only {p_value_exact*100:.1f}% of the time.',
        transform=ax.transAxes, va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='#FEF9E7', alpha=0.95))

blue_p = mpatches.Patch(color='#4A90D9', alpha=0.85, label='Scores within normal range for innocent students')
red_p  = mpatches.Patch(color='#E74C3C', alpha=0.85, label=f'p-value region — scores ≥ {observed_score} (only {p_value_exact*100:.1f}%)')
ax.legend(handles=[blue_p, red_p], fontsize=10)

ax.set_xlabel('Score on 10-Question Exam', fontsize=12)
ax.set_ylabel('Number of Students (out of 10,000)', fontsize=12)
ax.set_title('The p-value — How Rare Is a Score of 9 for an Innocent Student?',
             fontsize=13, fontweight='bold')
ax.set_xticks(score_vals)

plt.tight_layout()
plt.show()

---

## Step 6 — Make the Decision

We have:
- Our threshold: **α = 0.05**
- Our p-value:   **≈ 0.0107**

The decision rule is simple:

```
If p-value < α  →  Reject H₀  →  ⚖️ GUILTY
If p-value ≥ α  →  Fail to Reject H₀  →  ⚖️ NOT GUILTY
```

---

### ⚠️ Important: "Not Guilty" ≠ "Innocent"

Just like in a real court, we never say we **proved** the student is innocent. We only say:
- **Guilty** → the evidence is too strong to explain by luck alone
- **Not Guilty** → we don't have enough evidence to convict, but we're not saying they definitely didn't cheat

In [ ]:
# ── Step 6: Deliver the Verdict ─────────────────────────────

alpha = 0.05

print("=" * 52)
print("   ⚖️  THE VERDICT")
print("=" * 52)
print()
print(f"   p-value = {p_value_exact:.4f}")
print(f"   α       = {alpha}")
print()

if p_value_exact < alpha:
    print(f"   {p_value_exact:.4f} < {alpha}")
    print()
    print("   ✅ REJECT H₀")
    print()
    print("   ⚖️  GUILTY")
    print("   The score of 9/10 is too high to explain by")
    print("   random guessing. The evidence is suspicious")
    print("   enough to conclude the student was cheating.")
else:
    print(f"   {p_value_exact:.4f} ≥ {alpha}")
    print()
    print("   ❌ FAIL TO REJECT H₀")
    print()
    print("   ⚖️  NOT GUILTY")
    print("   Not enough evidence to conclude cheating.")

print("=" * 52)

In [ ]:
# ── Step 6 Visual: The verdict with threshold line ──────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('⚖️ Does the Evidence Cross the Threshold?', fontsize=14, fontweight='bold')

for ax, alpha_val, label in zip(
        axes,
        [0.05, 0.01],
        ['α = 0.05  (Standard threshold)', 'α = 0.01  (Stricter threshold)']):

    for s, c in zip(score_vals, counts):
        color = '#E74C3C' if s >= observed_score else '#4A90D9'
        ax.bar(s, c, color=color, alpha=0.75, edgecolor='white', linewidth=1.5, width=0.7)

    ax.axvline(x=observed_score - 0.5, color='darkred', linestyle='-', linewidth=3,
               label=f'Our student = {observed_score}/10')

    verdict = '✅ GUILTY\n(p < α)' if p_value_exact < alpha_val else '❌ NOT GUILTY\n(p ≥ α)'
    vc = '#27AE60' if p_value_exact < alpha_val else '#E74C3C'
    vb = '#D5F5E3' if p_value_exact < alpha_val else '#FADBD8'

    ax.text(0.5, 0.91, verdict, transform=ax.transAxes, ha='center', va='top',
            fontsize=13, fontweight='bold', color=vc,
            bbox=dict(boxstyle='round,pad=0.5', facecolor=vb, edgecolor=vc, linewidth=2))

    ax.set_title(f'{label}\n'
                 f'p-value = {p_value_exact:.4f}  {"<" if p_value_exact < alpha_val else "≥"}  α = {alpha_val}',
                 fontweight='bold')
    ax.set_xlabel('Score out of 10')
    ax.set_ylabel('Count')
    ax.set_xticks(score_vals)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("Key insight: the SAME score leads to different verdicts depending on the threshold!")
print("This is why you must choose α before you see the data.")

---

## Bonus: What If the Score Were Different?

Let's see how the verdict changes for **every possible score** from 0 to 10.

This will show us:
- Which scores are "suspicious enough" to convict
- Which scores are within the normal range for innocent students

In [ ]:
# ── Bonus: p-value for every possible score ──────────────────

print("⚖️  What would the verdict be for every possible score?")
print("=" * 60)
print(f"   {'Score':>7} | {'p-value':>10} | {'Verdict (α=0.05)':>22}")
print("   " + "-" * 47)

for s in range(11):
    pv = 1 - stats.binom.cdf(s - 1, n=10, p=0.5)
    verdict = "✅ GUILTY" if pv < 0.05 else "❌ Not guilty"
    marker = "  ◄ Our student" if s == observed_score else ""
    print(f"   {s:>7} | {pv:>10.4f} | {verdict:<22}{marker}")

print()
print("Only scores of 9 or 10 are suspicious enough to convict at α = 0.05.")

In [ ]:
# ── Bonus Visual ─────────────────────────────────────────────

all_pvals = [1 - stats.binom.cdf(s - 1, n=10, p=0.5) for s in score_vals]

fig, ax = plt.subplots(figsize=(11, 6))

bar_colors = ['#E74C3C' if p < 0.05 else '#27AE60' for p in all_pvals]
ax.bar(score_vals, all_pvals, color=bar_colors, alpha=0.85,
       edgecolor='white', linewidth=1.5, width=0.7)

# Label each bar
for s, p in zip(score_vals, all_pvals):
    ax.text(s, p + 0.01, f'{p:.3f}', ha='center', fontsize=8.5,
            fontweight='bold', color='#2C3E50')

ax.axhline(y=0.05, color='black', linestyle='--', linewidth=2.5,
           label='α = 0.05 (conviction threshold)')
ax.scatter([observed_score], [p_value_exact], color='purple',
           s=200, zorder=10, label=f'Our student (score={observed_score}, p={p_value_exact:.3f})')

green_p = mpatches.Patch(color='#27AE60', alpha=0.85, label='Not guilty — p ≥ 0.05')
red_p   = mpatches.Patch(color='#E74C3C', alpha=0.85, label='Guilty — p < 0.05')
ax.legend(handles=[green_p, red_p,
          plt.Line2D([0],[0], color='black', lw=2, linestyle='--', label='α = 0.05'),
          plt.scatter([],[],  color='purple', s=100, label=f'Our student (score=9)')],
          fontsize=9)

ax.set_xlabel('Score on 10-Question Exam', fontsize=12)
ax.set_ylabel('p-value', fontsize=12)
ax.set_title('p-value for Every Possible Score\n'
             'Scores in the RED zone are suspicious enough to convict (at α = 0.05)',
             fontsize=13, fontweight='bold')
ax.set_xticks(score_vals)
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

---

## ❌ Three Things the p-value Does NOT Mean

These mistakes are extremely common — even among scientists!

---

### ❌ Mistake 1: "p = 0.011 means there's a 1.1% chance the student is innocent"

**Wrong.** The p-value does not tell you the probability the student is innocent.

**Correct:** p = 0.011 means: *"If the student were innocent, there's a 1.1% chance they'd score this high by luck."*

⚖️ The p-value is about **the evidence**, not about guilt or innocence itself.

---

### ❌ Mistake 2: "p ≥ 0.05 proves the student is innocent"

**Wrong.** A high p-value just means *we don't have enough evidence to convict*.

⚖️ "Not Guilty" ≠ "Innocent". Maybe the student cheated cleverly and only got 7/10 on purpose so they wouldn't look suspicious!

---

### ❌ Mistake 3: "A significant result means the cheating was huge"

**Wrong.** Statistical significance just says the result is unlikely to be random chance. It says nothing about *how much* the student cheated.

A student who got 9/10 and a student who got 10/10 are both "guilty" — but they cheated by different amounts. The p-value doesn't tell you which is worse.

---

---

## 🔄 Type I & Type II Errors — When the Verdict Is Wrong

Even with a fair system, two types of mistakes can happen:

| | **Student was INNOCENT** | **Student was GUILTY** |
|---|---|---|
| **We say Guilty** | 🚨 Type I Error — Wrongful conviction | ✅ Correct |
| **We say Not Guilty** | ✅ Correct | ⚠️ Type II Error — Guilty walks free |

---

### 🚨 Type I Error — Convicting an Innocent Student

The student really **was** guessing randomly, but happened to score 9/10 by pure luck (it can happen — about 1.1% of the time!). We convict them unfairly.

- The probability of this happening = **α** (our threshold)
- By setting α = 0.05, we accept at most a **5% chance** of wrongfully convicting an innocent student

### ⚠️ Type II Error — Letting a Cheater Go Free

The student **was** cheating, but only scored 7/10 (maybe they were careful not to get too many right). We don't have enough evidence — they go free.

- The probability of this = **β**
- Reduced by: using a longer exam (more questions = more evidence = harder to hide cheating)

### ⚡ The Tradeoff

> If we lower α (harder to convict) → Fewer innocent students punished, but more cheaters escape  
> If we raise α (easier to convict) → Fewer cheaters escape, but more innocent students punished

In [ ]:
# ── Type I & II Error Visualization ─────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('🔄 Two Ways the Verdict Can Be Wrong', fontsize=14, fontweight='bold')

score_range = np.arange(0, 11)
h0_pmf = stats.binom.pmf(score_range, n=10, p=0.5)   # Innocent: guessing
h1_pmf = stats.binom.pmf(score_range, n=10, p=0.85)  # Guilty: knows answers
cutoff = 9  # reject H0 if score >= 9

# LEFT: Type I — Innocent student convicted
ax1 = axes[0]
for s, p in zip(score_range, h0_pmf):
    color = '#E74C3C' if s >= cutoff else '#4A90D9'
    ax1.bar(s, p, color=color, alpha=0.8, edgecolor='white', linewidth=1.5, width=0.7)

type1 = sum(h0_pmf[s] for s in score_range if s >= cutoff)
ax1.set_title(f'🚨 Type I Error: Wrongful Conviction\n'
              f'Innocent student happens to score {cutoff}+\n'
              f'Probability = {type1:.4f} ({type1*100:.1f}%) = α',
              fontweight='bold', color='#C0392B')
ax1.set_xlabel('Score out of 10')
ax1.set_ylabel('Probability')
ax1.set_xticks(score_range)
blue_p = mpatches.Patch(color='#4A90D9', alpha=0.8, label='Not convicted (correct)')
red_p  = mpatches.Patch(color='#E74C3C', alpha=0.8, label=f'Wrongly convicted (Type I = {type1:.3f})')
ax1.legend(handles=[blue_p, red_p], fontsize=9)

# RIGHT: Type II — Cheating student goes free
ax2 = axes[1]
for s, p in zip(score_range, h1_pmf):
    color = '#27AE60' if s >= cutoff else '#F39C12'
    ax2.bar(s, p, color=color, alpha=0.8, edgecolor='white', linewidth=1.5, width=0.7)

type2 = sum(h1_pmf[s] for s in score_range if s < cutoff)
power = 1 - type2
ax2.set_title(f'⚠️ Type II Error: Cheater Goes Free\n'
              f'Guilty student scores below {cutoff} — not caught\n'
              f'Probability = {type2:.4f} ({type2*100:.1f}%) = β',
              fontweight='bold', color='#E67E22')
ax2.set_xlabel('Score out of 10')
ax2.set_ylabel('Probability')
ax2.set_xticks(score_range)
orange_p = mpatches.Patch(color='#F39C12', alpha=0.8, label=f'Not caught — goes free (Type II = {type2:.3f})')
green_p  = mpatches.Patch(color='#27AE60', alpha=0.8, label=f'Correctly caught (Power = {power:.3f})')
ax2.legend(handles=[orange_p, green_p], fontsize=9)

plt.tight_layout()
plt.show()

print(f"Type I Error  (α):  {type1:.4f} — chance of wrongly convicting an innocent student")
print(f"Type II Error (β):  {type2:.4f} — chance of letting a cheating student go free")
print(f"Power (1−β):        {power:.4f} — chance of correctly catching a cheating student")

---

## 🏆 Full Summary — The Complete Story

---

In [ ]:
# ── Complete Flowchart ───────────────────────────────────────

fig, ax = plt.subplots(figsize=(13, 13))
ax.set_xlim(0, 10)
ax.set_ylim(0, 13)
ax.axis('off')
fig.patch.set_facecolor('#F8F9FA')
ax.set_facecolor('#F8F9FA')

ax.text(5, 12.5,
        '⚖️ Did the Student Cheat?\nThe Complete Hypothesis Testing Story',
        ha='center', va='center', fontsize=15, fontweight='bold', color='#2C3E50')

def box(ax, x, y, w, h, text, fc, tc='white', fs=10):
    r = mpatches.FancyBboxPatch((x-w/2, y-h/2), w, h,
                                 boxstyle='round,pad=0.15',
                                 facecolor=fc, edgecolor='white', linewidth=2, zorder=3)
    ax.add_patch(r)
    ax.text(x, y, text, ha='center', va='center', fontsize=fs,
            fontweight='bold', color=tc, zorder=4, multialignment='center')

def arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#7F8C8D', lw=2.5), zorder=2)

steps = [
    (5, 11.5, 7, 0.7,
     'STEP 1: Set the Hypotheses\nH₀: Student was guessing  |  H₁: Student was cheating',
     '#2980B9'),
    (5, 10.3, 7, 0.7,
     'STEP 2: Set the Threshold  (α = 0.05)\n"How suspicious must the score be to convict?"',
     '#8E44AD'),
    (5, 9.1, 7, 0.7,
     'STEP 3: Observe the Evidence\nStudent scored 9 out of 10',
     '#16A085'),
    (5, 7.9, 7, 0.7,
     'STEP 4: Build Null Distribution\nSimulate 10,000 innocent guessing students',
     '#D35400'),
    (5, 6.7, 7, 0.7,
     'STEP 5: Calculate p-value\nOnly 1.1% of innocent students score 9+ by luck',
     '#C0392B'),
    (5, 5.5, 7, 0.7,
     'STEP 6: Compare p-value to α\np = 0.011  <  α = 0.05',
     '#2C3E50'),
]

for x, y, w, h, text, color in steps:
    box(ax, x, y, w, h, text, color, fs=10)
for i in range(len(steps)-1):
    arrow(ax, 5, steps[i][1]-steps[i][3]/2-0.02,
             5, steps[i+1][1]+steps[i+1][3]/2+0.02)

# Verdict boxes
box(ax, 2.5, 4.0, 4.0, 1.2,
    '✅ REJECT H₀\n\n⚖️ GUILTY\nStudent was cheating',
    '#27AE60', fs=11)
box(ax, 7.5, 4.0, 4.0, 1.2,
    '❌ FAIL TO REJECT H₀\n\n⚖️ NOT GUILTY\nNot enough evidence',
    '#E74C3C', fs=11)

ax.annotate('', xy=(2.5, 4.6), xytext=(3.8, 5.18),
            arrowprops=dict(arrowstyle='->', color='#27AE60', lw=2.5))
ax.annotate('', xy=(7.5, 4.6), xytext=(6.2, 5.18),
            arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=2.5))
ax.text(3.0, 4.98, 'p < α', fontsize=12, color='#27AE60', fontweight='bold')
ax.text(6.5, 4.98, 'p ≥ α', fontsize=12, color='#E74C3C', fontweight='bold')

# Error labels
box(ax, 2.5, 2.5, 4.0, 1.0,
    '🚨 Risk: Type I Error\nWrongly convicted an innocent student\nProbability = α = 5%',
    '#FADBD8', '#C0392B', fs=9)
box(ax, 7.5, 2.5, 4.0, 1.0,
    '⚠️ Risk: Type II Error\nA cheating student goes free\nProbability = β',
    '#FDEBD0', '#784212', fs=9)

# Footer warning
ax.text(5, 1.3,
        '⚠️  p-value is NOT the probability the student is innocent\n'
        '      "Not Guilty" ≠ "Innocent"     |     Set α BEFORE looking at the data',
        ha='center', va='center', fontsize=9.5, style='italic', color='#7F8C8D')

plt.tight_layout()
plt.show()

---

## 📚 Quick Reference Card

| Courtroom | Hypothesis Testing |
|-----------|--------------------|
| Defendant (student) | Null Hypothesis H₀ |
| "Innocent until proven guilty" | H₀ assumed true until data says otherwise |
| Evidence (the score: 9/10) | Data collected |
| "Beyond reasonable doubt" | Significance level α |
| Judge evaluates evidence | Statistical test |
| Guilty verdict | Reject H₀ |
| Not Guilty verdict | Fail to Reject H₀ |
| Wrongful conviction | Type I Error (probability = α) |
| Guilty person goes free | Type II Error (probability = β) |

---

### The p-value in one sentence:

> **"If the student truly guessed randomly, what is the probability they would score this high — or higher — just by luck?"**

---

### The decision rule:

```
p-value < α   →   Reject H₀   →   GUILTY
p-value ≥ α   →   Fail to Reject H₀   →   NOT GUILTY
```

---

### p-value cheat sheet:

```
p < 0.01   →  Very suspicious  →  "Open and shut case"
p < 0.05   →  Suspicious       →  "Beyond reasonable doubt"
p < 0.10   →  Slightly odd     →  "Hmm, worth watching..."
p ≥ 0.10   →  Normal           →  "No case here"
```

---

## 🌟 The Big Idea

> Hypothesis testing doesn't tell you the **truth**.
>
> It tells you whether the **evidence is strong enough** to act as if something is true.
>
> Just like a court verdict — it's not about perfect knowledge. It's about making a **reasonable decision** given the evidence available. ⚖️